# 00 - Archive ingestion

Extract dated source ZIPs and append their CSV/Parquet files to
`archived.archived_<table>`. Every archived row receives a timestamp
`export_date` plus file/run lineage so `02a_archive_silver 02 03.ipynb` can replay
the exports into the shared `silver.slv_<table>` targets.

The process is restartable at ZIP and file level through global
`monitoring.cfg_*` tables. A successful item is skipped unless its `reload`
flag is set to `true`; failed or interrupted items are retried.

`export_date` is parsed from the originating ZIP filename (for example, `2026-06-29.zip` becomes `2026-06-29 00:00:00`), not from the CSV filename or the notebook run time.


In [ ]:
ARCHIVE_ZIP_ROOT = "Files/wmpp-production-data-export-birmingham/archive"
EXTRACT_ROOT = "Files/archive_unzipped"
ERROR_LOG_ROOT = "Files/archive_error_logs"
ARCHIVE_SCHEMA = "archived"
TABLE_PREFIX = "archived_"
MONITORING_SCHEMA = "monitoring"

PROCESS_EXPORT_DATE = ""  # Optional YYYY-MM-DD; blank processes all ZIPs chronologically.
RESET_ARCHIVE_TABLES = False  # Optional full reset; not required when export_date already exists.
FAIL_ON_FILE_ERROR = True
TEXT_QUALIFIER = '"'


In [ ]:
import os
import re
import shutil
import uuid
import zipfile
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType,
    TimestampType,
)

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
PIPELINE_NAME = "00_archive_load"


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def sql_string(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def append_rows(table_name, rows, schema):
    if rows:
        (spark.createDataFrame(rows, schema)
            .write.format("delta").mode("append").saveAsTable(table_name))


def parse_export_date(value):
    match = re.search(r"(\d{4}-\d{2}-\d{2})", value or "")
    if not match:
        return None
    return datetime.strptime(match.group(1), "%Y-%m-%d")


def safe_extract(zip_path, destination):
    destination_abs = os.path.abspath(destination)
    with zipfile.ZipFile(zip_path, "r") as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        for member in members:
            target = os.path.abspath(os.path.join(destination_abs, member.filename))
            if os.path.commonpath([destination_abs, target]) != destination_abs:
                raise ValueError(f"Unsafe ZIP member path: {member.filename}")
        archive.extractall(destination_abs)
    return len(members)


def clean_table_name(file_name):
    raw_name = os.path.splitext(os.path.basename(file_name))[0]
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", raw_name):
        raw_name = "audit"
    safe_name = re.sub(r"[^A-Za-z0-9_]", "_", raw_name).strip("_")
    if not safe_name:
        raise ValueError(f"Could not derive a table name from {file_name}")
    return f"{TABLE_PREFIX}{safe_name}"


ZIP_AUDIT_SCHEMA = StructType([
    StructField("zip_path", StringType(), False),
    StructField("export_date", TimestampType(), False),
    StructField("extract_path", StringType(), True),
    StructField("status", StringType(), False),
    StructField("reload", BooleanType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("file_count", IntegerType(), True),
    StructField("run_id", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("first_loaded_at", TimestampType(), True),
    StructField("last_updated_at", TimestampType(), True),
])

FILE_AUDIT_SCHEMA = StructType([
    StructField("file_path", StringType(), False),
    StructField("filename", StringType(), False),
    StructField("export_date", TimestampType(), False),
    StructField("source_zip", StringType(), True),
    StructField("target_object", StringType(), True),
    StructField("status", StringType(), False),
    StructField("reload", BooleanType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("run_id", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("first_loaded_at", TimestampType(), True),
    StructField("last_updated_at", TimestampType(), True),
])


def audit_record(table_name, key_columns):
    frame = spark.table(table_name)
    predicate = None
    for name, value in key_columns.items():
        condition = F.col(name) == F.lit(value).cast(frame.schema[name].dataType)
        predicate = condition if predicate is None else predicate & condition
    rows = frame.where(predicate).limit(1).collect()
    return rows[0].asDict() if rows else None


def merge_audit(table_name, schema, row, key_names):
    source = spark.createDataFrame([row], schema)
    condition = " AND ".join(f"t.{name} = s.{name}" for name in key_names)
    (DeltaTable.forName(spark, table_name).alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(ARCHIVE_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(MONITORING_SCHEMA)}")

spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_zip_load (
  zip_path STRING, export_date TIMESTAMP, extract_path STRING, status STRING,
  reload BOOLEAN, attempt_count INT, file_count INT, run_id STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, error_message STRING,
  first_loaded_at TIMESTAMP, last_updated_at TIMESTAMP
) USING DELTA
''')
spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_file_load (
  file_path STRING, filename STRING, export_date TIMESTAMP, source_zip STRING,
  target_object STRING, status STRING, reload BOOLEAN, attempt_count INT,
  rows_read BIGINT, rows_written BIGINT, run_id STRING, started_at TIMESTAMP,
  ended_at TIMESTAMP, error_message STRING, first_loaded_at TIMESTAMP,
  last_updated_at TIMESTAMP
) USING DELTA
''')
spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT,
  rows_written BIGINT, error_message STRING
) USING DELTA
''')
spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, rejected_row_count BIGINT, recorded_at TIMESTAMP
) USING DELTA
''')

append_rows(
    "monitoring.cfg_pipeline_run",
    [(RUN_ID, PIPELINE_NAME, "ARCHIVE", "ARCHIVE_ZIP", STARTED_AT, None,
      "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string",
)


In [ ]:
zip_root_posix = f"/lakehouse/default/{ARCHIVE_ZIP_ROOT}"
extract_root_posix = f"/lakehouse/default/{EXTRACT_ROOT}"
os.makedirs(extract_root_posix, exist_ok=True)

zip_batches = []
for root, _, files in os.walk(zip_root_posix):
    for file_name in files:
        if not file_name.lower().endswith(".zip"):
            continue
        full_path = os.path.join(root, file_name)
        relative_path = os.path.relpath(full_path, "/lakehouse/default").replace("\\", "/")
        export_date = parse_export_date(file_name) or parse_export_date(relative_path)
        if export_date is None:
            print(f"Skipping ZIP without YYYY-MM-DD export date: {relative_path}")
            continue
        if PROCESS_EXPORT_DATE and export_date.strftime("%Y-%m-%d") != PROCESS_EXPORT_DATE:
            continue
        zip_batches.append((export_date, relative_path, full_path))

zip_batches.sort(key=lambda item: (item[0], item[1]))
if not zip_batches:
    spark.sql(f'''UPDATE monitoring.cfg_pipeline_run
        SET ended_at=current_timestamp(), status='FAILED', tables_failed=1,
            error_message='No dated archive ZIP files found'
        WHERE run_id={sql_string(RUN_ID)}''')
    raise ValueError(f"No dated ZIP files found under {ARCHIVE_ZIP_ROOT}")
if RESET_ARCHIVE_TABLES and PROCESS_EXPORT_DATE:
    spark.sql(f'''UPDATE monitoring.cfg_pipeline_run
        SET ended_at=current_timestamp(), status='FAILED', tables_failed=1,
            error_message='RESET_ARCHIVE_TABLES requires all export dates'
        WHERE run_id={sql_string(RUN_ID)}''')
    raise ValueError(
        "RESET_ARCHIVE_TABLES=True must be run with PROCESS_EXPORT_DATE blank "
        "so every historical export is rebuilt."
    )

zip_errors = []
for export_date, relative_zip, full_zip in zip_batches:
    extract_relative = f"{EXTRACT_ROOT}/{export_date:%Y-%m-%d}"
    extract_posix = f"/lakehouse/default/{extract_relative}"
    existing = audit_record(
        "monitoring.cfg_archive_zip_load",
        {"zip_path": relative_zip, "export_date": export_date},
    )
    if existing and existing["status"] == "SUCCESS" and not existing["reload"]:
        print(f"Skipped extracted ZIP: {relative_zip}")
        continue

    now = datetime.utcnow()
    attempt = int(existing["attempt_count"] or 0) + 1 if existing else 1
    first_loaded = existing.get("first_loaded_at") if existing else None
    running = (relative_zip, export_date, extract_relative, "RUNNING",
               bool(existing["reload"]) if existing else False, attempt, None,
               RUN_ID, now, None, None, first_loaded, now)
    merge_audit("monitoring.cfg_archive_zip_load", ZIP_AUDIT_SCHEMA, running,
                ["zip_path", "export_date"])
    try:
        os.makedirs(extract_posix, exist_ok=True)
        file_count = safe_extract(full_zip, extract_posix)
        ended = datetime.utcnow()
        success = (relative_zip, export_date, extract_relative, "SUCCESS", False,
                   attempt, file_count, RUN_ID, now, ended, None,
                   first_loaded or ended, ended)
        merge_audit("monitoring.cfg_archive_zip_load", ZIP_AUDIT_SCHEMA, success,
                    ["zip_path", "export_date"])
        print(f"Extracted {relative_zip}: {file_count:,} files")
    except Exception as exc:
        ended = datetime.utcnow()
        error = str(exc)[:4000]
        failed = (relative_zip, export_date, extract_relative, "FAILED",
                  bool(existing["reload"]) if existing else False, attempt, None,
                  RUN_ID, now, ended, error, first_loaded, ended)
        merge_audit("monitoring.cfg_archive_zip_load", ZIP_AUDIT_SCHEMA, failed,
                    ["zip_path", "export_date"])
        zip_errors.append(f"{relative_zip}: {error}")
        if FAIL_ON_FILE_ERROR:
            break


In [ ]:
files_to_load = []
for zip_export_date, relative_zip, _ in zip_batches:
    extract_relative = f"{EXTRACT_ROOT}/{zip_export_date:%Y-%m-%d}"
    extract_posix = f"/lakehouse/default/{extract_relative}"
    if not os.path.isdir(extract_posix):
        continue
    for root, _, files in os.walk(extract_posix):
        for file_name in files:
            if not file_name.lower().endswith((".csv", ".parquet")):
                continue
            full_path = os.path.join(root, file_name)
            relative_path = os.path.relpath(full_path, "/lakehouse/default").replace("\\", "/")
            files_to_load.append((zip_export_date, relative_zip, relative_path, full_path, file_name))

files_to_load.sort(key=lambda item: (item[0], item[2]))
reset_targets = set()
file_errors = []
processed = skipped = total_read = total_written = 0

for expected_export_date, relative_zip, relative_path, full_path, file_name in files_to_load:
    # Derive the row-level timestamp from the ZIP that produced this file.
    export_date = parse_export_date(os.path.basename(relative_zip))
    if export_date is None:
        raise ValueError(f"Source ZIP has no YYYY-MM-DD date: {relative_zip}")
    if export_date != expected_export_date:
        raise ValueError(
            f"ZIP/export folder date mismatch for {relative_path}: "
            f"ZIP={export_date:%Y-%m-%d}, folder={expected_export_date:%Y-%m-%d}"
        )
    physical_table = clean_table_name(file_name)
    target_object = f"{ARCHIVE_SCHEMA}.{physical_table}"
    target_exists = spark.catalog.tableExists(target_object)
    target_columns = spark.table(target_object).columns if target_exists else []
    existing_export_rows = 0
    if target_exists and "export_date" in target_columns:
        existing_export_rows = (spark.table(target_object)
            .where(F.to_date("export_date") == F.lit(export_date.date()))
            .count())
    existing = audit_record(
        "monitoring.cfg_archive_file_load",
        {"file_path": relative_path, "export_date": export_date},
    )
    if (existing and existing["status"] == "SUCCESS"
            and not existing["reload"] and not RESET_ARCHIVE_TABLES):
        skipped += 1
        print(f"Skipped loaded file: {relative_path}")
        continue

    # Existing dated rows are already replay-ready. If their older load did
    # not retain source-path lineage, infer a successful file audit and do
    # not append the same export again.
    if (existing_export_rows > 0 and "_archive_source_path" not in target_columns
            and not (existing and existing.get("reload"))
            and not RESET_ARCHIVE_TABLES):
        now = datetime.utcnow()
        attempt = int(existing["attempt_count"] or 0) + 1 if existing else 1
        first_loaded = existing.get("first_loaded_at") if existing else now
        inferred = (relative_path, file_name, export_date, relative_zip, target_object,
                    "SUCCESS", False, attempt, existing_export_rows, 0, RUN_ID,
                    now, now, "Inferred from existing archive export_date rows",
                    first_loaded, now)
        merge_audit("monitoring.cfg_archive_file_load", FILE_AUDIT_SCHEMA, inferred,
                    ["file_path", "export_date"])
        skipped += 1
        print(f"Skipped existing dated archive rows: {target_object} @ {export_date:%Y-%m-%d}")
        continue

    now = datetime.utcnow()
    attempt = int(existing["attempt_count"] or 0) + 1 if existing else 1
    first_loaded = existing.get("first_loaded_at") if existing else None
    running = (relative_path, file_name, export_date, relative_zip, target_object,
               "RUNNING", bool(existing["reload"]) if existing else False,
               attempt, None, None, RUN_ID, now, None, None, first_loaded, now)
    merge_audit("monitoring.cfg_archive_file_load", FILE_AUDIT_SCHEMA, running,
                ["file_path", "export_date"])

    try:
        if file_name.lower().endswith(".parquet"):
            frame = spark.read.format("parquet").load(relative_path)
        else:
            frame = (spark.read.format("csv")
                .option("header", "true")
                .option("inferSchema", "false")
                .option("mode", "PERMISSIVE")
                .option("badRecordsPath", f"{ERROR_LOG_ROOT}/corrupt_rows")
                .option("quote", TEXT_QUALIFIER)
                .option("escape", TEXT_QUALIFIER)
                .option("multiLine", "true")
                .load(relative_path))

        frame = (frame
            .withColumn(
                "export_date",
                F.to_timestamp(F.lit(export_date.strftime("%Y-%m-%d")), "yyyy-MM-dd"),
            )
            .withColumn("_archive_source_path", F.lit(relative_path))
            .withColumn("_archive_source_zip", F.lit(relative_zip))
            .withColumn("_archive_run_id", F.lit(RUN_ID))
            .withColumn("_archive_load_ts", F.current_timestamp()))
        row_count = frame.count()

        if RESET_ARCHIVE_TABLES and target_object not in reset_targets:
            spark.sql(f"DROP TABLE IF EXISTS {qident(ARCHIVE_SCHEMA)}.{qident(physical_table)}")
            reset_targets.add(target_object)

        if spark.catalog.tableExists(target_object):
            target_columns = spark.table(target_object).columns
            if "_archive_source_path" in target_columns:
                DeltaTable.forName(spark, target_object).delete(
                    F.col("_archive_source_path") == F.lit(relative_path)
                )
            elif "export_date" not in target_columns:
                raise ValueError(
                    f"{target_object} has neither export_date nor source-file lineage."
                )
            elif existing and existing.get("reload"):
                if physical_table.lower() == "archived_audit":
                    raise ValueError(
                        "A legacy archived_audit file cannot be replaced individually "
                        "without _archive_source_path."
                    )
                DeltaTable.forName(spark, target_object).delete(
                    F.to_date("export_date") == F.lit(export_date.date())
                )

        (frame.write.format("delta").mode("append")
            .option("mergeSchema", "true").saveAsTable(target_object))
        ended = datetime.utcnow()
        success = (relative_path, file_name, export_date, relative_zip, target_object,
                   "SUCCESS", False, attempt, row_count, row_count, RUN_ID, now,
                   ended, None, first_loaded or ended, ended)
        merge_audit("monitoring.cfg_archive_file_load", FILE_AUDIT_SCHEMA, success,
                    ["file_path", "export_date"])
        append_rows(
            "monitoring.cfg_table_load_metric",
            [(RUN_ID, "ARCHIVE", "ARCHIVE_FILE", relative_path, target_object,
              row_count, row_count, 0, None, ended)],
            "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,rejected_row_count long,recorded_at timestamp",
        )
        processed += 1
        total_read += row_count
        total_written += row_count
        print(f"Loaded {relative_path} -> {target_object}: {row_count:,} rows")
    except Exception as exc:
        ended = datetime.utcnow()
        error = str(exc)[:4000]
        failed = (relative_path, file_name, export_date, relative_zip, target_object,
                  "FAILED", bool(existing["reload"]) if existing else False,
                  attempt, None, None, RUN_ID, now, ended, error, first_loaded, ended)
        merge_audit("monitoring.cfg_archive_file_load", FILE_AUDIT_SCHEMA, failed,
                    ["file_path", "export_date"])
        file_errors.append(f"{relative_path}: {error}")
        print(f"FAILED {relative_path}: {error}")
        if FAIL_ON_FILE_ERROR:
            break


In [ ]:
all_errors = zip_errors + file_errors
status = "FAILED" if all_errors else "SUCCESS"
error_text = " | ".join(all_errors)[:4000] if all_errors else None
spark.sql(f'''
UPDATE monitoring.cfg_pipeline_run
SET ended_at = current_timestamp(),
    status = {sql_string(status)},
    tables_succeeded = {processed},
    tables_failed = {len(all_errors)},
    rows_read = {total_read},
    rows_written = {total_written},
    error_message = {sql_string(error_text)}
WHERE run_id = {sql_string(RUN_ID)}
''')

print(
    f"Archive ingestion {status}: processed={processed}, skipped={skipped}, "
    f"failed={len(all_errors)}, rows={total_written:,}"
)
if all_errors:
    raise RuntimeError(error_text)
